## WP003 — Bookmaker Benchmark

See `README.md` for methodology and rationale. This notebook reuses WP001's 401 walk-forward CV match predictions verbatim (no retraining), joins historical closing odds from football-data.co.uk, and compares the model to the market: pooled RPS, paired bootstrap vs. Pinnacle, calibration, and edge-detection on disagreement / promoted-team subsets.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq

from football_model.model.predict import dc_outcome_probs

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'

with open(WP001 / 'cv_checkpoint.pkl', 'rb') as f:
    cp = pickle.load(f)
with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

df_cv = shared['df_cv']
windows = shared['windows']
match_preds = cp['cv_match_predictions']
print(len(match_preds), 'model predictions across', len({m['window'] for m in match_preds}), 'windows')

401 model predictions across 35 windows


### 1. Attach (date, teams, round) to each model prediction

The checkpoint stores only `window / lambda_home / lambda_away / goals_home / goals_away / rho_dc` per match. Rebuild the fixture identity by replaying the *exact* row selection `scripts/run_cv_window.py` does: sort `df_cv` by datetime, keep `is_home==1` rows, filter to the window's test rounds. Zip against the per-window predictions and assert the stored score equals the fixture's actual score — a hard check we've lined up the right matches.

In [2]:
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)
first_round = df_cv.groupby('season')['round'].min().to_dict()

# team code -> football-data.co.uk name (only 6 differ from the plain long name)
CODE_TO_FD = {
    'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace',
    'EVE': 'Everton', 'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds',
    'LEI': 'Leicester', 'LIV': 'Liverpool', 'LUT': 'Luton', 'MCI': 'Man City',
    'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich', 'NOT': "Nott'm Forest",
    'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland', 'TOT': 'Tottenham',
    'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves',
}

rows = []
for w in sorted({m['window'] for m in match_preds}):
    win = windows[w - 1]
    sel = df_sorted[
        (df_sorted['is_home'] == 1)
        & (df_sorted['round'] >= win['test_start'])
        & (df_sorted['round'] <= win['test_end'])
    ]
    wp = [m for m in match_preds if m['window'] == w]
    assert len(sel) == len(wp), (w, len(sel), len(wp))
    for (_, r), m in zip(sel.iterrows(), wp):
        assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away'], \
            (w, r['team'], r['opp_team'])
        rows.append({
            'window': w,
            'date': pd.Timestamp(r['datetime']).normalize(),
            'season': r['season'],
            'abs_round': int(r['round']),
            'season_round': int(r['round']) - first_round[r['season']] + 1,
            'home_code': r['team'], 'away_code': r['opp_team'],
            'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
            'goals_home': m['goals_home'], 'goals_away': m['goals_away'],
            'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'],
            'rho_dc': m.get('rho_dc'),
        })

model_df = pd.DataFrame(rows)
probs = [dc_outcome_probs(r.lambda_home, r.lambda_away, rho=r.rho_dc) for r in model_df.itertuples()]
model_df[['p_home_model', 'p_draw_model', 'p_away_model']] = np.array(probs)
model_df['result'] = np.where(model_df['goals_home'] > model_df['goals_away'], 'H',
                       np.where(model_df['goals_home'] == model_df['goals_away'], 'D', 'A'))
print(len(model_df), 'matches;', model_df['date'].min().date(), '->', model_df['date'].max().date())
model_df.head()

401 matches; 2021-09-11 -> 2026-05-18


,window,date,season,abs_round,season_round,home_code,away_code,home_fd,away_fd,goals_home,goals_away,lambda_home,lambda_away,rho_dc,p_home_model,p_draw_model,p_away_model,result
0,1,2021-09-11,2021,37,4,CRY,TOT,Crystal Palace,Tottenham,3,0,0.991480,1.743706,0.033728,0.215530,0.228079,0.556391,H
1,1,2021-09-11,2021,37,4,SOU,WHU,Southampton,West Ham,0,0,1.244713,1.875610,0.033728,0.255156,0.218373,0.526471,D
2,1,2021-09-11,2021,37,4,WAT,WOL,Watford,Wolves,0,2,1.151595,0.890184,0.033728,0.422783,0.292039,0.285178,A
3,1,2021-09-11,2021,37,4,BRE,BRI,Brentford,Brighton,0,1,1.067556,0.785787,0.033728,0.422426,0.309455,0.268119,A
4,1,2021-09-11,2021,37,4,MUN,NEW,Man United,Newcastle,4,1,2.166726,0.968639,0.033728,0.652568,0.188843,0.158588,H


### 2. Fetch closing odds from football-data.co.uk

One CSV per season (`mmz4281/<code>/E0.csv`). Cached to `odds_raw.pkl` — delete that file to re-fetch.

In [3]:
ODDS_CACHE = WP003 / 'odds_raw.pkl'
SEASON_CODES = ['2021', '2122', '2223', '2324', '2425', '2526']  # 2020-21 ... 2025-26

if ODDS_CACHE.exists():
    odds_raw = pd.read_pickle(ODDS_CACHE)
    print('loaded cached odds:', len(odds_raw), 'matches')
else:
    frames = []
    for code_ in SEASON_CODES:
        url = f'https://www.football-data.co.uk/mmz4281/{code_}/E0.csv'
        try:
            odf = pd.read_csv(url, encoding='latin-1')
            odf['season_code'] = code_
            frames.append(odf)
            print(code_, '->', len(odf), 'matches')
        except Exception as e:
            print(code_, 'FAILED:', repr(e))
    odds_raw = pd.concat(frames, ignore_index=True)
    odds_raw['Date'] = pd.to_datetime(odds_raw['Date'], dayfirst=True).dt.normalize()
    odds_raw.to_pickle(ODDS_CACHE)
    print('fetched + cached:', len(odds_raw), 'matches')

BOOK_COLS = {
    'pinnacle': [('PSCH', 'PSCD', 'PSCA'), ('PSH', 'PSD', 'PSA')],
    'b365':     [('B365CH', 'B365CD', 'B365CA'), ('B365H', 'B365D', 'B365A')],
    'avg':      [('AvgCH', 'AvgCD', 'AvgCA'), ('AvgH', 'AvgD', 'AvgA')],
    'max':      [('MaxCH', 'MaxCD', 'MaxCA'), ('MaxH', 'MaxD', 'MaxA')],
}
BOOK_PICK = {}
for book, options in BOOK_COLS.items():
    for cols in options:
        if all(c in odds_raw.columns for c in cols):
            BOOK_PICK[book] = cols
            break
print('\nusing columns:', BOOK_PICK)

2021 -> 380 matches
2122 -> 380 matches
2223 -> 380 matches
2324 -> 380 matches
2425 -> 380 matches
2526 -> 380 matches
fetched + cached: 2280 matches

using columns: {'pinnacle': ('PSCH', 'PSCD', 'PSCA'), 'b365': ('B365CH', 'B365CD', 'B365CA'), 'avg': ('AvgCH', 'AvgCD', 'AvgCA'), 'max': ('MaxCH', 'MaxCD', 'MaxCA')}


### 3. Join model predictions to odds

Join on `(date, home_fd, away_fd)`. Report any unmatched fixtures — iterate on `CODE_TO_FD` if needed. Assert joined actual score agrees with football-data's `FTHG`/`FTAG`.

In [4]:
j = model_df.merge(
    odds_raw, left_on=['date', 'home_fd', 'away_fd'],
    right_on=['Date', 'HomeTeam', 'AwayTeam'], how='left', indicator=True,
)
matched = j[j['_merge'] == 'both'].copy()
unmatched = j[j['_merge'] == 'left_only']
print(f'{len(matched)}/{len(j)} model predictions matched to odds; {len(unmatched)} unmatched')
if len(unmatched):
    print(unmatched[['date', 'home_fd', 'away_fd', 'season']].to_string(index=False))

bad = matched[(matched['goals_home'] != matched['FTHG']) | (matched['goals_away'] != matched['FTAG'])]
print(f'score mismatches after join: {len(bad)} (must be 0)')
assert len(bad) == 0

401/401 model predictions matched to odds; 0 unmatched
score mismatches after join: 0 (must be 0)


### 4. De-vig odds → implied probabilities

`proportional`: normalise `1/odds` to sum to 1. `shin`: Shin (1993), correcting favourite-longshot bias (solved with a 1-D root find on the informed-trader fraction z). Both added for every available book; downstream uses `proportional` by default.

In [5]:
def devig_proportional(o):
    inv = 1.0 / np.asarray(o, float)
    return inv / inv.sum()

def devig_shin(o):
    inv = 1.0 / np.asarray(o, float)
    B = inv.sum()
    def probs(z):
        return (np.sqrt(z * z + 4 * (1 - z) * inv * inv / B) - z) / (2 * (1 - z))
    if inv.sum() <= 1.0:
        return inv / inv.sum()
    try:
        z = brentq(lambda z: probs(z).sum() - 1.0, 1e-9, 0.5)
    except ValueError:
        z = 0.0
    p = probs(z)
    return p / p.sum()

def add_probs(dfm, book, cols, method, suffix):
    P = np.array([method(row) for row in dfm[list(cols)].to_numpy()])
    dfm[f'p_home_{book}{suffix}'] = P[:, 0]
    dfm[f'p_draw_{book}{suffix}'] = P[:, 1]
    dfm[f'p_away_{book}{suffix}'] = P[:, 2]

for book, cols in BOOK_PICK.items():
    ok = matched[list(cols)].notna().all(axis=1)
    matched[f'overround_{book}'] = np.nan
    matched.loc[ok, f'overround_{book}'] = (1 / matched.loc[ok, cols[0]] + 1 / matched.loc[ok, cols[1]] + 1 / matched.loc[ok, cols[2]]) - 1.0
    sub = matched.loc[ok]
    add_probs(matched, book, cols, devig_proportional, '')          # p_home_pinnacle etc.
    add_probs(matched, book, cols, devig_shin, '_shin')             # p_home_pinnacle_shin etc.
    print(f'{book:<9} n={int(ok.sum()):>3}  mean overround {matched[f"overround_{book}"].mean():.2%}')

pinnacle  n=361  mean overround 2.78%
b365      n=401  mean overround 5.49%
avg       n=401  mean overround 4.30%
max       n=401  mean overround -0.53%


### 5. Pooled RPS — model vs. each book vs. naive

In [6]:
def rps_hda(ph, pd_, pa, actual):
    cp1, cp2 = ph, ph + pd_
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)

def bootstrap_mean_ci(values, n_boot=5000, alpha=0.05, seed=0):
    v = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(v, size=len(v), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(bm, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return v.mean(), lo, hi

def rps_series(dfm, name):
    return dfm.apply(lambda x: rps_hda(x[f'p_home_{name}'], x[f'p_draw_{name}'], x[f'p_away_{name}'], x['result']), axis=1)

avg_h, avg_a = df_cv['goals_home'].mean(), df_cv['goals_away'].mean()
naive_probs = dc_outcome_probs(avg_h, avg_a)

names = ['model'] + [b for b in ['pinnacle', 'b365', 'avg'] if f'p_home_{b}' in matched.columns]
rows = []
for name in names:
    sub = matched.dropna(subset=[f'p_home_{name}'])
    r = rps_series(sub, name)
    m, lo, hi = bootstrap_mean_ci(r.values)
    rows.append({'predictor': name, 'n': len(r), 'pooled_rps': round(m, 4), 'ci': f'[{lo:.4f}, {hi:.4f}]'})
nr = matched['result'].map(lambda a: rps_hda(*naive_probs, a))
m, lo, hi = bootstrap_mean_ci(nr.values)
rows.append({'predictor': 'naive', 'n': len(nr), 'pooled_rps': round(m, 4), 'ci': f'[{lo:.4f}, {hi:.4f}]'})
print(pd.DataFrame(rows).to_string(index=False))

predictor   n  pooled_rps               ci
    model 401      0.1963 [0.1851, 0.2081]
 pinnacle 361      0.1786 [0.1659, 0.1919]
     b365 401      0.1833 [0.1714, 0.1958]
      avg 401      0.1834 [0.1713, 0.1960]
    naive 401      0.2341 [0.2289, 0.2391]


### 6. Paired bootstrap — model minus Pinnacle, per match

Positive = model has higher (worse) RPS than the sharp line. The question is how far positive, and how tight the CI.

In [7]:
if 'p_home_pinnacle' in matched.columns:
    s = matched.dropna(subset=['p_home_pinnacle']).copy()
    rm = rps_series(s, 'model').values
    rp = rps_series(s, 'pinnacle').values
    diff = rm - rp
    m, lo, hi = bootstrap_mean_ci(diff)
    verdict = ('model reliably WORSE than Pinnacle' if lo > 0 else
               'model reliably BETTER than Pinnacle' if hi < 0 else
               'not distinguishable from Pinnacle')
    print(f'n = {len(s)}')
    print(f'mean RPS(model) - RPS(Pinnacle): {m:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]')
    print(f'relative to Pinnacle RPS:        {m / rp.mean():+.1%}')
    print('->', verdict)
    # sensitivity: Shin de-vig instead of proportional
    if 'p_home_pinnacle_shin' in matched.columns:
        rp_shin = s.apply(lambda x: rps_hda(x['p_home_pinnacle_shin'], x['p_draw_pinnacle_shin'], x['p_away_pinnacle_shin'], x['result']), axis=1).values
        m2, lo2, hi2 = bootstrap_mean_ci(rm - rp_shin)
        print(f'(Shin de-vig: {m2:+.4f}  CI [{lo2:+.4f}, {hi2:+.4f}])')
else:
    print('Pinnacle columns not available')

n = 361
mean RPS(model) - RPS(Pinnacle): +0.0139   95% CI [+0.0080, +0.0198]
relative to Pinnacle RPS:        +7.8%
-> model reliably WORSE than Pinnacle
(Shin de-vig: +0.0142  CI [+0.0083, +0.0202])


### 7. Calibration — model vs. market

The market is near-perfectly calibrated by construction; where the model's curve departs from it on the same matches is where it's losing.

In [8]:
def reliability_table(p, y, n_bins=8):
    p, y = np.asarray(p, float), np.asarray(y, float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        msk = idx == b
        if msk.sum() == 0:
            continue
        out.append({'bin': f'{edges[b]:.2f}-{edges[b+1]:.2f}', 'n': int(msk.sum()),
                    'pred': round(p[msk].mean(), 3), 'actual': round(y[msk].mean(), 3)})
    return pd.DataFrame(out)

y_home = (matched['result'] == 'H').astype(int).values
print('MODEL  — predicted P(home win) vs actual:')
print(reliability_table(matched['p_home_model'].values, y_home).to_string(index=False))
if 'p_home_pinnacle' in matched.columns:
    s = matched.dropna(subset=['p_home_pinnacle'])
    print('\nPINNACLE — predicted P(home win) vs actual:')
    print(reliability_table(s['p_home_pinnacle'].values, (s['result'] == 'H').astype(int).values).to_string(index=False))

MODEL  — predicted P(home win) vs actual:
      bin   n  pred  actual
0.00-0.12   7 0.103   0.000
0.12-0.25  64 0.195   0.172
0.25-0.38  96 0.316   0.354
0.38-0.50 114 0.437   0.465
0.50-0.62  65 0.560   0.523
0.62-0.75  40 0.669   0.750
0.75-0.88  14 0.782   0.929
0.88-1.00   1 0.877   1.000

PINNACLE — predicted P(home win) vs actual:
      bin  n  pred  actual
0.00-0.12 23 0.093   0.043
0.12-0.25 59 0.200   0.203
0.25-0.38 73 0.310   0.329
0.38-0.50 79 0.438   0.430
0.50-0.62 62 0.563   0.532
0.62-0.75 42 0.687   0.881
0.75-0.88 22 0.807   0.818
0.88-1.00  1 0.916   1.000


### 8. Edge detection — where the model disagrees with the market

Sort by `|P_model(home) - P_market(home)|`. On the matches where the model most disagrees with the sharp line, is the model's RPS better or worse than the market's? Better on the tail = a real, isolable edge.

In [9]:
mkt = 'pinnacle' if 'p_home_pinnacle' in matched.columns else ('avg' if 'p_home_avg' in matched.columns else None)
if mkt is None:
    print('no market probs available')
else:
    d = matched.dropna(subset=[f'p_home_{mkt}']).copy()
    d['disagreement'] = (d['p_home_model'] - d[f'p_home_{mkt}']).abs()
    d['rps_model'] = rps_series(d, 'model').values
    d['rps_mkt'] = rps_series(d, mkt).values
    print(f'market = {mkt}, n = {len(d)}\n')
    for label, q in [('all matches', 0.0), ('top 50% disagreement', 0.5), ('top 25%', 0.75), ('top 10%', 0.90)]:
        sub = d[d['disagreement'] >= d['disagreement'].quantile(q)]
        gap = sub['rps_model'].mean() - sub['rps_mkt'].mean()
        _, lo, hi = bootstrap_mean_ci((sub['rps_model'] - sub['rps_mkt']).values)
        flag = '   <-- model better here' if hi < 0 else ''
        print(f'{label:<22} n={len(sub):>3}  model {sub["rps_model"].mean():.4f}  '
              f'market {sub["rps_mkt"].mean():.4f}  gap {gap:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

market = pinnacle, n = 361

all matches            n=361  model 0.1925  market 0.1786  gap +0.0139  CI [+0.0080, +0.0198]
top 50% disagreement   n=181  model 0.2030  market 0.1798  gap +0.0232  CI [+0.0124, +0.0339]
top 25%                n= 91  model 0.2045  market 0.1659  gap +0.0386  CI [+0.0203, +0.0556]
top 10%                n= 37  model 0.2208  market 0.1797  gap +0.0411  CI [+0.0069, +0.0744]


### 9. Promoted-team subset

Teams in a season but not the one before. Bookmakers are historically soft on them early. Model vs. market RPS on matches involving a newly-promoted side, split by early season (rounds 1-10) vs. later.

In [10]:
season_teams = {s: set(g['team']) | set(g['opp_team']) for s, g in df_cv.groupby('season')}
ss = sorted(season_teams)
promoted = {s: (season_teams[s] - season_teams[ss[i - 1]]) if i > 0 else set() for i, s in enumerate(ss)}
print('promoted per season:', {s: sorted(v) for s, v in promoted.items() if v})

if mkt:
    d = matched.dropna(subset=[f'p_home_{mkt}']).copy()
    d['rps_model'] = rps_series(d, 'model').values
    d['rps_mkt'] = rps_series(d, mkt).values
    d['has_promoted'] = [
        (hc in promoted.get(s, set())) or (ac in promoted.get(s, set()))
        for hc, ac, s in zip(d['home_code'], d['away_code'], d['season'])
    ]
    print()
    for label, sub in [
        ('promoted, rounds 1-10', d[d['has_promoted'] & (d['season_round'] <= 10)]),
        ('promoted, round 11+',   d[d['has_promoted'] & (d['season_round'] > 10)]),
        ('no promoted team',      d[~d['has_promoted']]),
    ]:
        if len(sub) == 0:
            print(f'{label:<22} n=0'); continue
        gap = sub['rps_model'].mean() - sub['rps_mkt'].mean()
        _, lo, hi = bootstrap_mean_ci((sub['rps_model'] - sub['rps_mkt']).values)
        flag = '   <-- model better' if hi < 0 else ''
        print(f'{label:<22} n={len(sub):>3}  model {sub["rps_model"].mean():.4f}  '
              f'market {sub["rps_mkt"].mean():.4f}  gap {gap:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

promoted per season: {'2021': ['BRE', 'NOR', 'WAT'], '2022': ['BOU', 'FLH', 'NOT'], '2023': ['BUR', 'LUT', 'SHE'], '2024': ['IPS', 'LEI', 'SOU'], '2025': ['BUR', 'LED', 'SUN']}

promoted, rounds 1-10  n= 29  model 0.1626  market 0.1349  gap +0.0277  CI [+0.0109, +0.0456]
promoted, round 11+    n= 79  model 0.1660  market 0.1508  gap +0.0152  CI [+0.0037, +0.0257]
no promoted team       n=253  model 0.2042  market 0.1923  gap +0.0120  CI [+0.0048, +0.0193]


### 10. Toy betting sim (illustrative only)

Flat 1-unit stake on any outcome where `P_model` exceeds the best-available (`Max`) implied probability by threshold τ, settled at the `Max` price. ROI over a τ sweep with a bootstrap CI on per-bet profit. **In-sample to the τ choice** — a diagnostic for "is there anything here at all", not a strategy.

In [11]:
if 'max' in BOOK_PICK:
    mcols = BOOK_PICK['max']
    sim = matched.dropna(subset=list(mcols)).copy()
    legs = []
    for _, x in sim.iterrows():
        for out_, pcol, ocol in [('H', 'p_home_model', mcols[0]), ('D', 'p_draw_model', mcols[1]), ('A', 'p_away_model', mcols[2])]:
            legs.append({'edge': x[pcol] - 1.0 / x[ocol], 'odds': x[ocol], 'won': x['result'] == out_})
    legs = pd.DataFrame(legs)
    print(f'{"tau":>6} {"n_bets":>7} {"roi":>9} {"95% CI":>24}')
    for tau in [0.0, 0.02, 0.05, 0.08, 0.10, 0.15]:
        b = legs[legs['edge'] >= tau]
        if len(b) == 0:
            print(f'{tau:>6.2f} {0:>7}'); continue
        profit = np.where(b['won'], b['odds'] - 1.0, -1.0)
        _, lo, hi = bootstrap_mean_ci(profit)
        print(f'{tau:>6.2f} {len(b):>7} {profit.mean():>+9.1%}   [{lo:+.2%}, {hi:+.2%}]')
else:
    print('no Max columns available')

   tau  n_bets       roi                   95% CI
  0.00     582    -16.9%   [-31.11%, -1.43%]
  0.02     429    -23.0%   [-39.70%, -5.40%]
  0.05     264    -24.4%   [-43.70%, -3.17%]
  0.08     126    -27.6%   [-55.95%, +5.43%]
  0.10      82    -29.4%   [-66.12%, +17.87%]
  0.15      19    -52.1%   [-100.00%, +16.59%]
